# Notebook 02 — Fine-tune Student Model
**Member: B** | Runtime > Change runtime type > **T4 GPU** before running.

| Experiment | `data.mode` | `training.output_dir` |
|---|---|---|
| Exp 2 (adaptation)   | `adaptation`   | `results/checkpoints/adaptation` |
| Exp 3 (distillation) | `distillation` | `results/checkpoints/distillation` |
| Exp 4 (filtered)     | `filtered`     | `results/checkpoints/filtered` |

Ablations — change one field in `configs/finetune.yaml` per run:
- A2 rank sweep: `lora.rank` ∈ {4, 8, 16, 32}
- A3 data size: `data.train_fraction` ∈ {0.1, 0.25, 0.5, 1.0}

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import subprocess
from pathlib import Path
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token

repo = Path('/content/SC4001-Group133')
repo_url = 'https://github.com/TIFFMUN/SC4001-Group133.git'

if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', repo_url, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)

os.chdir(repo)
%pip install -q -r requirements.txt
%pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sc4001-group133 (pyproject.toml) ... done


In [3]:
# Drive cache and project paths
import importlib
importlib.invalidate_caches()
from src.setup_cache import setup
setup()

# Use a fresh local model-download cache. Keep trained checkpoints on Drive.
# This applies when the training script uses Hugging Face's default cache.
cache_dir = '/content/sc4001_hf_cache'
os.environ['HF_HOME'] = cache_dir
os.environ['HF_HUB_CACHE'] = f'{cache_dir}/hub'
os.environ['HF_XET_CACHE'] = f'{cache_dir}/xet'
os.environ.pop('TRANSFORMERS_CACHE', None)

print('HF cache:', os.environ['HF_HUB_CACHE'])

[setup_cache] Cache redirected to Drive: /content/drive/MyDrive/SC4001
HF cache: /content/sc4001_hf_cache/hub


In [4]:
# Link processed data and checkpoints to Google Drive
from pathlib import Path
from datetime import datetime
import shutil

project = Path('/content/SC4001-Group133')
drive_root = Path('/content/drive/MyDrive/SC4001')

for name in ['all', 'filtered']:
    source = drive_root / 'data/processed' / name
    if not (source / 'dataset_dict.json').is_file():
        raise FileNotFoundError(f'Dataset not found: {source}')

def link_folder(local, target):
    target.mkdir(parents=True, exist_ok=True)
    if local.resolve() == target.resolve():
        return
    if local.is_symlink():
        raise RuntimeError(f'{local} already links to another location')
    if local.exists():
        if not local.is_dir():
            raise RuntimeError(f'Unexpected file: {local}')
        if any(local.iterdir()):
            backup = drive_root / 'backups' / f'{local.name}_{datetime.now():%Y%m%d_%H%M%S_%f}'
            backup.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(local), str(backup))
            print('Preserved previous local folder:', backup)
        else:
            local.rmdir()
    local.parent.mkdir(parents=True, exist_ok=True)
    local.symlink_to(target, target_is_directory=True)

link_folder(project / 'data/processed', drive_root / 'data/processed')
link_folder(project / 'results', drive_root / 'results')
print('Data:', (project / 'data/processed').resolve())
print('Results:', (project / 'results').resolve())

Data: /content/drive/MyDrive/SC4001/data/processed
Results: /content/drive/MyDrive/SC4001/results


In [5]:
# GPU
import torch
assert torch.cuda.is_available(), 'Enable T4 GPU in Runtime > Change runtime type'
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

GPU: Tesla T4 | VRAM: 15.6 GB


In [6]:
# Select experiment 3 or 4
import json
import yaml

experiment = 4  # Change to 3 only if you need to rerun Experiment 3
modes = {3: 'distillation', 4: 'filtered'}
mode = modes[experiment]

with open('configs/finetune.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['mode'] = mode
cfg['training']['output_dir'] = f'results/checkpoints/{mode}'

with open('configs/finetune.yaml', 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

checkpoint_dir = Path(cfg['training']['output_dir'])
adapter_present = any((checkpoint_dir / name).is_file()
                      for name in ['adapter_model.safetensors', 'adapter_model.bin'])

states = list(checkpoint_dir.glob('checkpoint-*/trainer_state.json'))
training_complete = False
if states and adapter_present:
    latest = max(states, key=lambda p: int(p.parent.name.split('-')[-1]))
    with open(latest) as f:
        state = json.load(f)
    expected = state.get('max_steps')
    actual = state.get('global_step')
    target_epochs = cfg['training']['num_epochs']
    training_complete = (expected is not None and actual == expected and
                         (state.get('epoch') or 0) >= target_epochs - 0.01)

print(f"Exp {experiment} | Mode: {mode} | Rank: {cfg['lora']['rank']} | Epochs: {cfg['training']['num_epochs']}")
print('Output:', checkpoint_dir.resolve())
print('Final adapter present:', adapter_present, '| Training verified complete:', training_complete)

if adapter_present and not training_complete:
    raise RuntimeError('A final adapter exists, but completion is unverified. Inspect it before retraining to avoid overwriting it.')

Exp 4 | Mode: filtered | Rank: 16 | Epochs: 3
Output: /content/drive/MyDrive/SC4001/results/checkpoints/filtered
Final adapter present: False | Training verified complete: False


In [7]:
# Train selected experiment
# If a cached model weight is malformed, identify and repair that cache file only.
import gc
from datetime import datetime
from pathlib import Path
from safetensors import SafetensorError
import transformers.modeling_utils as modeling_utils
from huggingface_hub import hf_hub_download
from src.training.finetune import train

original_safe_open = modeling_utils.safe_open
failed_weights = []

def track_safe_open(filename, *args, **kwargs):
    try:
        return original_safe_open(filename, *args, **kwargs)
    except SafetensorError:
        failed_weights.append(Path(filename))
        raise

def repair_cached_weight(filename):
    # Only modify a Hugging Face Hub model cache, never project checkpoints.
    parts = filename.parts
    model_parts = [(i, part) for i, part in enumerate(parts) if part.startswith('models--')]
    if not model_parts:
        print('Bad weight file:', filename)
        print('This is not a recognizable Hugging Face model-cache file; no file was changed.')
        return False

    i, model_folder = model_parts[-1]
    if 'snapshots' not in parts[i+1:]:
        print('Could not identify the model snapshot:', filename)
        return False

    snap_i = parts.index('snapshots', i+1)
    if snap_i+2 >= len(parts):
        return False

    repo_id = model_folder.removeprefix('models--').replace('--', '/')
    revision = parts[snap_i+1]
    relative_name = '/'.join(parts[snap_i+2:])
    hub_cache = Path(*parts[:i])

    # Rename the broken blob instead of deleting it. Do not touch results/.
    blob = filename.resolve()
    if not blob.is_file() or 'blobs' not in blob.parts:
        print('Unexpected cache layout; no file changed:', filename)
        return False
    if hub_cache.resolve() not in blob.parents:
        print('File is outside the expected HF cache; no file changed:', blob)
        return False
    backup = blob.with_name(blob.name + '.invalid_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
    blob.rename(backup)
    print('Moved invalid cached model weight to:', backup)
    print('Downloading a clean copy:', repo_id, relative_name)
    hf_hub_download(repo_id=repo_id, filename=relative_name, revision=revision,
                    cache_dir=str(hub_cache), force_download=True, token=hf_token)
    return True

if training_complete:
    print(f'Experiment {experiment} is already complete; skipping training.')
else:
    modeling_utils.safe_open = track_safe_open
    try:
        try:
            train()
        except SafetensorError:
            if not failed_weights or not repair_cached_weight(failed_weights[-1]):
                raise
            print('Retrying training after repairing the cached weight...')
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            train()  # Retry once only
    finally:
        modeling_utils.safe_open = original_safe_open

W1115 08:08:59.862000 5881 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1115 08:08:59.902000 5881 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (13 outputs left out)

Building labels for eval dataset:   0%|          | 0/9 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/9 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/9 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
15,No log,0.413913,0.344317,0.895112,102480.000000


In [8]:
# Confirm checkpoint in Google Drive
checkpoint_dir = Path(cfg['training']['output_dir'])
print('Checkpoint:', checkpoint_dir.resolve())
if checkpoint_dir.exists():
    print('Saved files:', sorted(p.name for p in checkpoint_dir.iterdir()))
    print('Final adapter saved:', any((checkpoint_dir / name).is_file()
                                      for name in ['adapter_model.safetensors', 'adapter_model.bin']))
    states = list(checkpoint_dir.glob('checkpoint-*/trainer_state.json'))
    if states:
        last = max(states, key=lambda p: int(p.parent.name.split('-')[-1]))
        with open(last) as f:
            state = json.load(f)
        print('Epoch:', state.get('epoch'), '| Steps:', state.get('global_step'), '/', state.get('max_steps'))
else:
    print('No checkpoint found. Inspect the training error above.')

Checkpoint: /content/drive/MyDrive/SC4001/results/checkpoints/filtered
Saved files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'checkpoint-15', 'run_config.yaml', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']
Final adapter saved: True
Epoch: 3.0 | Steps: 15 / 15
